In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Key landmarks from mediapipe indices
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_HIP = 23
RIGHT_HIP = 24

In [3]:
def normalize_pose(df):
    """
    Input: dataframe with columns
    [frame, landmark, x, y, z, visibility]

    Output: normalized coordinates
    """

    normalized_data = []

    grouped = df.groupby("frame")

    for frame_id, frame_data in grouped:

        # Extract needed joints
        try:
            l_shoulder = frame_data[frame_data["landmark"] == LEFT_SHOULDER].iloc[0]
            r_shoulder = frame_data[frame_data["landmark"] == RIGHT_SHOULDER].iloc[0]
            l_hip = frame_data[frame_data["landmark"] == LEFT_HIP].iloc[0]
            r_hip = frame_data[frame_data["landmark"] == RIGHT_HIP].iloc[0]
        except:
            continue  # skip frame if missing

        # --- 1. Compute hip center (origin) ---
        hip_center_x = (l_hip["x"] + r_hip["x"]) / 2
        hip_center_y = (l_hip["y"] + r_hip["y"]) / 2

        # --- 2. Compute scale (torso length) ---
        shoulder_center_x = (l_shoulder["x"] + r_shoulder["x"]) / 2
        shoulder_center_y = (l_shoulder["y"] + r_shoulder["y"]) / 2

        torso_length = np.sqrt(
            (shoulder_center_x - hip_center_x) ** 2 +
            (shoulder_center_y - hip_center_y) ** 2
        )

        if torso_length == 0:
            continue

        # --- 3. Normalize all landmarks ---
        for _, row in frame_data.iterrows():

            norm_x = (row["x"] - hip_center_x) / torso_length
            norm_y = (row["y"] - hip_center_y) / torso_length

            normalized_data.append({
                "frame": frame_id,
                "landmark": row["landmark"],
                "x": norm_x,
                "y": norm_y,
                "z": row["z"],  # optional
                "visibility": row["visibility"]
            })

    return pd.DataFrame(normalized_data)

In [4]:
df = pd.read_csv("../outputs/keypoints/push_up_keypoints.csv")

normalized_df = normalize_pose(df)

normalized_df.to_csv("../outputs/keypoints/push_up_normalized.csv", index=False)

print("Normalization complete!")

Normalization complete!
